# Advanced video delta compression demo

This notebook builds a synthetic natural-image video sequence, models each frame with piecewise bicubic patches,
encodes only frame-to-frame deltas as MPS/TT states, and measures:

1. **Bond dimension vs. reconstruction accuracy** for compressed deltas.
2. **Transmission cost** (bytes/frame) for sending compressed updates.

The workflow mirrors the 2D webcam notebook but adds a full compression study.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import quimb.tensor as qtn

from skimage import data
from skimage.color import rgb2gray
from skimage.transform import resize, rotate

from tensor_stream.piecewise import fit_bicubic_to_image, reconstruct_image_from_coefficients


In [ ]:
# -----------------------------
# Config
# -----------------------------
FRAME_SIZE = 64          # 64x64 => 2^12 sites for binary TT encoding
N_FRAMES = 24
PATCH_GRID_K = 4         # 2^k x 2^k patches (k=4 => 16x16 patches, each 4x4)
BOND_DIMENSIONS = [2, 4, 8, 16, 32, 64]
DELTA_DTYPE_BYTES = 4    # float32 payload assumption
RNG_SEED = 7


In [ ]:
def build_natural_video(n_frames=N_FRAMES, frame_size=FRAME_SIZE, seed=RNG_SEED):
    """Generate a smooth motion sequence from a natural image."""
    rng = np.random.default_rng(seed)

    base = rgb2gray(data.astronaut())
    base = resize(base, (frame_size + 12, frame_size + 12), anti_aliasing=True)

    frames = []
    for t in range(n_frames):
        # Smooth camera-like motion (translation + mild rotation + gain drift)
        tx = int(3.5 * np.sin(2 * np.pi * t / n_frames))
        ty = int(2.5 * np.cos(2 * np.pi * t / n_frames))
        angle = 2.5 * np.sin(4 * np.pi * t / n_frames)
        gain = 1.0 + 0.06 * np.sin(2 * np.pi * t / max(2, n_frames // 2))

        rotated = rotate(base, angle=angle, resize=False, mode='reflect')
        shifted = np.roll(rotated, shift=(tx, ty), axis=(0, 1))
        crop = shifted[6:6 + frame_size, 6:6 + frame_size]

        noise = 0.01 * rng.standard_normal(crop.shape)
        frame = np.clip(gain * crop + noise, 0.0, 1.0)
        frames.append(frame.astype(np.float32))

    return np.stack(frames, axis=0)


video = build_natural_video()
T, H, W = video.shape
print(f"Video shape: {video.shape}")


In [ ]:
def piecewise_bicubic_reconstruction(video_frames, k=PATCH_GRID_K):
    """Fit per-frame piecewise bicubic model and reconstruct the approximated video."""
    recon = np.zeros_like(video_frames, dtype=np.float32)
    for t in range(video_frames.shape[0]):
        coeffs, _ = fit_bicubic_to_image(video_frames[t], k=k)
        img_hat, _ = reconstruct_image_from_coefficients(coeffs)
        recon[t] = img_hat.astype(np.float32)
    return recon


video_poly = piecewise_bicubic_reconstruction(video)

deltas = np.zeros_like(video_poly)
deltas[1:] = video_poly[1:] - video_poly[:-1]

print("Delta tensor:", deltas.shape)
print("Raw delta payload per update (float32):", H * W * DELTA_DTYPE_BYTES, "bytes")


In [ ]:
def compress_delta_as_mps(delta_frame, bond_dim):
    """Compress one 2D delta frame as an MPS with given max bond dimension."""
    dense_vec = delta_frame.reshape(-1)
    n_sites = int(np.log2(dense_vec.size))
    if 2 ** n_sites != dense_vec.size:
        raise ValueError("Frame size must be a power of two in total pixels.")

    mps = qtn.MatrixProductState.from_dense(dense_vec, dims=[2] * n_sites)
    mps.compress(max_bond=bond_dim)

    recon = mps.to_dense().reshape(delta_frame.shape).real.astype(np.float32)

    # approximate transport size: all tensor elements * dtype bytes
    n_scalars = sum(np.prod(ts.shape) for ts in mps.arrays)
    payload_bytes = int(n_scalars * DELTA_DTYPE_BYTES)

    return recon, payload_bytes


def mse(a, b):
    return float(np.mean((a - b) ** 2))


def psnr(a, b, data_range=1.0):
    e = mse(a, b)
    if e <= 1e-15:
        return 120.0
    return float(10 * np.log10((data_range ** 2) / e))


In [ ]:
# Evaluate bond dimension sweep over all delta updates
results = []

for chi in BOND_DIMENSIONS:
    recon_deltas = np.zeros_like(deltas)
    payloads = []

    for t in range(1, T):
        d_hat, n_bytes = compress_delta_as_mps(deltas[t], bond_dim=chi)
        recon_deltas[t] = d_hat
        payloads.append(n_bytes)

    # Reconstruct full video from first frame + cumulative deltas
    recon_video = np.zeros_like(video_poly)
    recon_video[0] = video_poly[0]
    for t in range(1, T):
        recon_video[t] = recon_video[t - 1] + recon_deltas[t]

    results.append({
        'bond_dim': chi,
        'delta_mse': mse(deltas[1:], recon_deltas[1:]),
        'delta_psnr': psnr(deltas[1:], recon_deltas[1:], data_range=np.ptp(deltas[1:]) + 1e-8),
        'video_mse_vs_poly': mse(video_poly, recon_video),
        'video_psnr_vs_poly': psnr(video_poly, recon_video),
        'mean_payload_bytes': float(np.mean(payloads)),
        'median_payload_bytes': float(np.median(payloads)),
    })

results


In [ ]:
chis = np.array([r['bond_dim'] for r in results])
video_psnr = np.array([r['video_psnr_vs_poly'] for r in results])
delta_psnr = np.array([r['delta_psnr'] for r in results])
mean_payload = np.array([r['mean_payload_bytes'] for r in results])

raw_payload = H * W * DELTA_DTYPE_BYTES
ratio = mean_payload / raw_payload

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(chis, video_psnr, 'o-', label='Video PSNR vs piecewise model')
axes[0].plot(chis, delta_psnr, 's--', label='Delta PSNR')
axes[0].set_xscale('log', base=2)
axes[0].set_xlabel('Max bond dimension')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('Accuracy vs bond dimension')
axes[0].grid(True, alpha=0.25)
axes[0].legend()

axes[1].plot(chis, mean_payload, 'o-')
axes[1].axhline(raw_payload, color='r', linestyle='--', label='Raw delta payload')
axes[1].set_xscale('log', base=2)
axes[1].set_yscale('log')
axes[1].set_xlabel('Max bond dimension')
axes[1].set_ylabel('Bytes per delta update')
axes[1].set_title('Compressed transport cost')
axes[1].grid(True, which='both', alpha=0.25)
axes[1].legend()

axes[2].plot(video_psnr, ratio, 'o-')
axes[2].set_xlabel('Video PSNR vs piecewise model (dB)')
axes[2].set_ylabel('Payload / raw delta payload')
axes[2].set_title('Accuracy-cost frontier')
axes[2].grid(True, alpha=0.25)

plt.tight_layout()


In [ ]:
# Compact text table (no extra dependencies)
header = f"{'chi':>5} {'delta_psnr':>12} {'video_psnr':>12} {'mean_bytes':>12} {'ratio_to_raw':>12}"
print(header)
print('-' * len(header))
for r in sorted(results, key=lambda x: x['mean_payload_bytes']):
    ratio = r['mean_payload_bytes'] / (H * W * DELTA_DTYPE_BYTES)
    print(f"{r['bond_dim']:5d} {r['delta_psnr']:12.3f} {r['video_psnr_vs_poly']:12.3f} {r['mean_payload_bytes']:12.1f} {ratio:12.4f}")


## What this demonstrates

- The **piecewise bicubic model** captures frame content in the same spirit as the webcam example.
- Sending only **compressed deltas** can drastically reduce update payload.
- The bond-dimension sweep gives a direct **accuracy vs bandwidth curve** for natural-image video dynamics.

You can tune:

- `FRAME_SIZE`, `PATCH_GRID_K`, and `N_FRAMES` to change workload.
- `BOND_DIMENSIONS` to explore more aggressive compression.
- `DELTA_DTYPE_BYTES` if you quantize deltas (e.g., fp16 or int8).
